# nanoPotential: a matter model in one file

A machine-learning interatomic potential (MLIP) is a function. It takes a list of atoms, each one an element and a position in space, and returns one number: the potential energy of that arrangement. That is the whole contract. Everything a molecular-dynamics engine needs beyond the energy, it gets by differentiating the energy.

The models in the Almanac (MACE, UMA, Orb, NequIP) all honor that contract. They differ in how they turn positions into a number, and those differences drive their accuracy and speed. You can hold the shape of the field in your head once you have built one small honest example.

About 150 lines of plain PyTorch, no `torch_geometric` and no `e3nn`, nothing you can't step through in a debugger. It trains, runs dynamics, and shows the one fact that reshapes MLIP infrastructure: **inference includes a backward pass.**

Runs on a Colab CPU runtime in about a minute. `Runtime → Run all`, then read top to bottom.

## The interface

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

N_ELEMENTS = 20    # H (Z=1) through Ca (Z=20)
CUTOFF     = 5.0   # angstroms
N_RBF      = 16    # radial basis functions per edge
HIDDEN     = 64
N_LAYERS   = 2

A batch is three tensors. `Z` is a long tensor of shape `[N]`, the atomic number of each atom. `R` is a float tensor of shape `[N, 3]`, Cartesian positions in ångströms. `batch` is a long tensor of shape `[N]` mapping each atom to the structure it belongs to.

That third tensor is how MLIPs batch. Structures have different atom counts, so you concatenate instead of padding. Ten structures of ~50 atoms become one 500-atom system, and the `batch` vector remembers the boundaries. Because interactions are local (next section), atoms from different structures never see each other. It is one big block-diagonal graph. If you have batched graphs before, this is that.

## Neighbors

An atom's energy contribution depends on its local environment. Two atoms 40 Å apart barely feel each other, so we impose a cutoff: each atom interacts only with neighbors inside 5 Å. This locality is what lets these models scale to millions of atoms, because cost then grows linearly with atom count instead of quadratically.

In [ ]:
def neighbor_pairs(R, batch, cutoff):
    # all (center i, neighbor j) pairs within cutoff, same structure only
    d = torch.cdist(R, R)                                   # [N, N]
    same = batch[:, None] == batch[None, :]
    mask = same & (d < cutoff) & (d > 1e-8)                 # drop self-pairs
    idx_i, idx_j = mask.nonzero(as_tuple=True)              # each: [E]
    return idx_i, idx_j

This is O(N²) in memory and time, fine for a toy with hundreds of atoms and hopeless for a million. Real codes bin atoms into cutoff-sized cells so each atom checks only the 27 adjacent cells, which makes neighbor search O(N). Real codes also handle periodic boundary conditions: a crystal is an infinite tiling of one repeating box, so a neighbor can be an atom's own periodic image. We skip both. Nothing downstream changes shape either way: the output is an edge list, `idx_i` and `idx_j`, each of length E.

Note the direction convention: `idx_i` is the receiving atom, `idx_j` the neighbor sending to it. Both `(i, j)` and `(j, i)` appear in the list, so every atom hears from every neighbor.

## Edge features, and why the envelope matters

A neighbor at 1.1 Å means something very different from one at 4.9 Å. We hand the network the distance, but not as a raw scalar. We expand it over a set of Gaussian bumps spaced along `[0, cutoff]`, so "how far away" becomes a 16-dimensional soft one-hot the MLP can use without first learning to bin distances itself.

Then we multiply by a smooth envelope that falls to zero at the cutoff. This is not cosmetic. The cutoff makes the energy a function with a boundary: a neighbor at 4.99 Å contributes, one at 5.01 Å does not. If the contribution did not taper to zero, an atom drifting across the boundary would make the energy jump discontinuously. Forces are the derivative of the energy, and the derivative of a jump is a spike. One atom crossing the cutoff would kick the simulation with a nonsense force, and molecular dynamics integrates every kick. The cosine envelope makes the contribution, and its derivative, go smoothly to zero at 5 Å, so atoms enter and leave neighborhoods without anyone feeling a seam.

In [ ]:
class RadialBasis(nn.Module):
    def __init__(self, n_rbf, cutoff):
        super().__init__()
        self.register_buffer("centers", torch.linspace(0.0, cutoff, n_rbf))
        self.gamma  = (n_rbf / cutoff) ** 2      # width ~ center spacing
        self.cutoff = cutoff

    def forward(self, d):                                    # d: [E]
        g = torch.exp(-self.gamma * (d[:, None] - self.centers) ** 2)  # [E, n_rbf]
        env = 0.5 * (torch.cos(torch.pi * d / self.cutoff) + 1.0)      # [E]
        return g * env[:, None]

## Message passing

Each atom starts as a vector looked up by element, from an embedding table of 20 rows. Identity enters here and carries little; the geometry does the work.

Then, twice, every atom updates its vector by listening to its neighbors. One round: for each edge, concatenate the neighbor's current features with the edge's radial features, push that through a small MLP to get a message, sum all messages arriving at each center atom, and add the transformed sum back residually.

The sum is the load-bearing choice. Addition is commutative, so an atom's update does not depend on the order its neighbors happen to appear in the edge list, which physics requires because neighbors have no order. The sum is a `scatter_add`: E messages land on N atoms according to `idx_i`. In plain PyTorch that is `index_add_`, and it is the same scatter pattern every graph network uses.

In [ ]:
class Interaction(nn.Module):
    def __init__(self, hidden, n_rbf):
        super().__init__()
        self.message = nn.Sequential(
            nn.Linear(hidden + n_rbf, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),
        )
        self.update = nn.Sequential(
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),
        )

    def forward(self, h, edge_feat, idx_i, idx_j):
        m = self.message(torch.cat([h[idx_j], edge_feat], dim=-1))   # [E, hidden]
        agg = torch.zeros_like(h).index_add_(0, idx_i, m)            # [N, hidden]
        return h + self.update(agg)

Two rounds means information hops two edges. An atom's final features can depend on atoms up to 10 Å away even though each round looks only 5 Å. The receptive field grows with depth, the way it does for stacked convolutions.

## Readout

After message passing, each atom's vector is a summary of its chemical environment. A small MLP maps it to one number: that atom's share of the energy. We add a per-element constant, a learnable reference energy, because a lone carbon atom carries a large baseline that has nothing to do with its surroundings, and making the network fit that offset wastes capacity. Then we sum atom energies within each structure.

The sum hard-codes a physical fact: two identical, well-separated molecules have exactly twice the energy of one. A per-atom sum gives you that for free, where an average would not.

In [ ]:
class NanoPotential(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(N_ELEMENTS + 1, HIDDEN)    # index by Z directly
        self.rbf = RadialBasis(N_RBF, CUTOFF)
        self.layers = nn.ModuleList(
            Interaction(HIDDEN, N_RBF) for _ in range(N_LAYERS))
        self.readout = nn.Sequential(
            nn.Linear(HIDDEN, HIDDEN // 2), nn.SiLU(),
            nn.Linear(HIDDEN // 2, 1),
        )
        self.e_ref = nn.Embedding(N_ELEMENTS + 1, 1)         # per-element baseline
        nn.init.zeros_(self.e_ref.weight)

    def forward(self, Z, R, batch):
        with torch.no_grad():          # neighbor *selection* is discrete
            idx_i, idx_j = neighbor_pairs(R, batch, CUTOFF)
        d = (R[idx_j] - R[idx_i]).norm(dim=-1)   # recomputed: gradients flow to R
        h = self.embed(Z)
        edge_feat = self.rbf(d)
        for layer in self.layers:
            h = layer(h, edge_feat, idx_i, idx_j)
        e_atom = self.readout(h).squeeze(-1) + self.e_ref(Z).squeeze(-1)   # [N]
        n_struct = int(batch.max()) + 1
        E = torch.zeros(n_struct, device=R.device).index_add_(0, batch, e_atom)
        return E                                              # [B]

Two lines earn their keep here. Which atoms count as neighbors is a discrete decision, so we find pairs under `no_grad`, but the distances fed to the network are recomputed from `R` on the differentiable path. That distinction is about to matter.

Notice what the model never sees: no bond list, no molecular graph from a chemist, no angles. Elements and positions in, one scalar out.

## Forces come out of the backward pass

A force is the negative gradient of the energy with respect to position: which way, and how hard, each atom is pushed. Classical potentials derive force expressions by hand. We have autograd.

In [ ]:
def energy_and_forces(model, Z, R, batch, training=False):
    R = R.requires_grad_(True)
    E = model(Z, R, batch)
    # E.sum() is safe: structure b's energy doesn't depend on structure c's atoms,
    # so the gradient lands on the right atoms regardless
    F = -torch.autograd.grad(E.sum(), R, create_graph=training)[0]   # [N, 3]
    return E, F

Two lines, two consequences that shape the infrastructure around these models.

**First: inference includes a backward pass.** Not for training, for the *product*. The force is a gradient, so producing it means backpropagating through the whole network on every call. When you serve a normal model you run forward only, and a stack of optimizations assumes that: `torch.no_grad`, cached activations, no autograd at serve time. An MLIP in production runs forward *and* backward on every step of every simulation. Plan roughly 2–3× the memory and time of a forward-only model of the same size; measure on your architecture. (Some models predict forces from a separate head instead, buying back forward-only inference at the cost of forces that are no longer exactly the energy's gradient, with real consequences for long runs. Orb-v2 did this; orb-v3 ships both; released UMA and eSEN are conservative.)

**Second: training differentiates the gradient.** The training loss includes a force term, and forces are already first derivatives of the energy. To get a gradient of the loss with respect to the *weights*, autograd differentiates through the force computation, a second-order derivative. That is what `create_graph=training` does: it makes the backward pass build its own graph, so it can be backpropagated through in turn. This is why force training costs roughly 2–3× an energy-only step (a separate 2–3× from the serving one: that was first-order forward-plus-backward, this is second-order), and why forgetting `create_graph=True` gives the classic silent bug: the loss drops while the force term contributes no weight gradients at all.

## A stand-in for DFT

Real training labels come from density functional theory, a quantum-mechanical calculation that takes minutes to hours of CPU per snapshot and returns one energy plus a force vector per atom. The big public datasets (OMat24, OMol25, MPtrj) hold hundreds of millions of such snapshots, bought with enormous compute. That expense is the reason MLIPs exist: pay for DFT once, then evaluate a million times faster.

We have no DFT budget, so we manufacture labels from a Morse potential, a classical two-atom energy curve with a repulsive wall, a well, and a decaying tail. It is a caricature of chemistry, but it is a smooth function of positions with real forces, which is all the training loop needs. Note the trick: we get the label forces by calling autograd on the *reference* function, the same move the model itself will use.

In [ ]:
def morse_labels(R, batch, D=1.0, a=1.5, r0=1.5):
    R = R.detach().requires_grad_(True)
    idx_i, idx_j = neighbor_pairs(R, batch, CUTOFF)
    d = (R[idx_j] - R[idx_i]).norm(dim=-1)
    pair = D * (1.0 - torch.exp(-a * (d - r0))) ** 2 - D
    n_struct = int(batch.max()) + 1
    # each pair appears as (i,j) and (j,i): halve to count bonds once
    E = torch.zeros(n_struct).index_add_(0, batch[idx_i], 0.5 * pair)
    F = -torch.autograd.grad(E.sum(), R)[0]
    return E.detach(), F.detach()

def sample_batch(n_struct=8, n_atoms=8, jitter=0.35):
    # perturbed 2x2x2 cubes of carbon: our "DFT snapshots"
    cube = torch.tensor([[float(x), float(y), float(z)]
                         for x in (0, 1) for y in (0, 1) for z in (0, 1)]) * 1.6
    R = cube.repeat(n_struct, 1) + jitter * torch.randn(n_struct * n_atoms, 3)
    Z = torch.full((n_struct * n_atoms,), 6, dtype=torch.long)
    batch = torch.arange(n_struct).repeat_interleave(n_atoms)
    E, F = morse_labels(R, batch)
    return Z, R, batch, E, F

## Training

The loss has two terms: energy error and force error. Convention: normalize the energy term per atom (a 500-atom structure shouldn't dominate a 20-atom one just by being bigger), and weight the force term heavily.

The heavy force weight is not arbitrary. Each snapshot gives one energy but 3N force components (24 numbers against 1, for our 8-atom cubes), and every force component measures how the energy *changes* locally, which is exactly the structure the model must get right to run dynamics. Forces carry most of the training signal; the energy term mostly pins down the offsets.

In [ ]:
model = NanoPotential()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
W_E, W_F = 1.0, 100.0

for step in range(2000):
    Z, R, batch, E_ref, F_ref = sample_batch()
    E, F = energy_and_forces(model, Z, R, batch, training=True)
    n_atoms = torch.bincount(batch).float()
    loss = W_E * ((E - E_ref) / n_atoms).pow(2).mean() \
         + W_F * (F - F_ref).pow(2).mean()
    opt.zero_grad()
    loss.backward()      # differentiates through the force computation
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)   # tiny random batches spike; clip
    opt.step()
    if step % 200 == 0:
        print(f"step {step:4d}  loss {loss.item():.5f}")

That `loss.backward()` is the second-order pass from the forces section. Profile a real MLIP training run and this is where the time goes.

## Serving: the MD loop

Molecular dynamics is Newton's law integrated numerically: read forces, nudge velocities, nudge positions, repeat. Velocity Verlet is the standard integrator; it splits the velocity update around the position update, which keeps energy from drifting the way plain Euler does.

We record three numbers each step: the potential energy the model reports, the kinetic energy from the velocities, and their sum. A constant-energy run should hold that sum flat, the acceptance test a static force error cannot give you.

In [ ]:
def run_md(model, Z, R, batch, m, dt=0.05, n_steps=2000, kick=0.15):
    # velocity Verlet; toy units, dt ~ a femtosecond in spirit
    torch.manual_seed(1)
    v = kick * torch.randn_like(R)              # a small thermal kick
    _, F = energy_and_forces(model, Z, R, batch)
    hist = []
    for step in range(n_steps):
        v = v + 0.5 * dt * F / m
        R = (R + dt * v).detach()               # detach: don't grow one giant graph
        E, F = energy_and_forces(model, Z, R, batch)
        v = v + 0.5 * dt * F / m
        KE = (0.5 * m * v.pow(2)).sum().item()
        PE = E.item()
        hist.append((step * dt, PE, KE, PE + KE))
    return torch.tensor(hist)                    # [n_steps, 4]: t, PE, KE, E_tot

Z, R, batch, _, _ = sample_batch(n_struct=1)
hist = run_md(model, Z, R, batch, m=torch.full((R.shape[0], 1), 12.0))
t, PE, KE, Etot = hist.T
print(f"total-energy drift over the run: {(Etot.max() - Etot.min()).item():.4f} "
      f"({100*(Etot.max()-Etot.min())/PE.abs().mean():.2f}% of mean |PE|)")

Plot the three curves. Potential and kinetic energy trade back and forth as the cluster breathes; their sum stays nearly flat. The residual wobble is integrator error; it shrinks if you shrink `dt`, and watching this sum is how you tune it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, PE,   label="potential",    lw=1.2)
ax.plot(t, KE,   label="kinetic",      lw=1.2)
ax.plot(t, Etot, label="total", color="k", lw=1.6)
ax.set_xlabel("time (toy units)"); ax.set_ylabel("energy")
ax.set_title("Energy over a constant-energy MD run"); ax.legend(loc="center right")
fig.tight_layout(); plt.show()

Every step is one model call, forward plus backward, and essentially nothing else. A real timestep is about a femtosecond; a nanosecond of physics is a million steps, a microsecond a billion. There is no batching across steps, because step N+1's input is step N's output. The model call *is* the workload. When you build inference tooling for MLIPs, this loop is what you're optimizing: latency per call, sustained, sequentially, millions of times, with a backward pass inside each one.

That is nanoPotential. Everything above is roughly 150 lines and runs on a laptop CPU in about a minute.

## The ladder: what the real ones add

Every production model in the Almanac is a set of deltas against the file you just ran. Here they are, rung by rung, each pointing at the line it replaces.

**Angles and directions.** Our edge features come from `d = (R[idx_j] - R[idx_i]).norm(dim=-1)`, and the norm throws away the direction, so a neighborhood is just a bag of distances. Two arrangements with identical distance lists but different angles (three neighbors in a row versus three at 90°) are indistinguishable to our `Interaction` layer, and they have genuinely different energies. Directional models keep the unit vector we discarded and expand it in spherical harmonics, the angular analog of our radial Gaussians, so messages carry "where" as well as "how far." DimeNet and GemNet did this with explicit angle terms; everything after does it with harmonics.

**Equivariant features.** Once directions enter, the hidden state can't stay a plain vector of scalars, because scalars don't know what happens when you rotate the molecule. Equivariant models replace our `h: [N, HIDDEN]` with typed channels (scalars, 3-vectors, higher moments) that transform correctly under rotation by construction. Mixing types (a vector times a vector can yield a scalar, a vector, and a rank-2 part) is done with Clebsch–Gordan tensor products, the machinery `e3nn` packages and NequIP built on. That tensor product replaces the `torch.cat` + MLP inside our `message`, and it is the hot kernel of the whole model class, precisely what NVIDIA's cuEquivariance exists to accelerate. If you're profiling MACE or NequIP on GPU, you're profiling tensor products.

**Body order.** Our message MLP sees one neighbor at a time; the atom only learns about neighbor *combinations* through the sum, slowly, across layers. MACE's move is to build symmetrized products of the aggregated neighborhood features, so each layer captures several neighbors jointly: genuine many-body terms per layer instead of pairwise terms stacked. The payoff: MACE gets away with two message-passing rounds (the same `N_LAYERS = 2` as ours) while matching models that need more, because each round is so much more expressive.

**Attention.** Our aggregation is `index_add_`, every neighbor's message summed with equal standing, weighted only by what the message MLP baked in. Transformer-style MLIPs (EquiformerV2, eSEN) replace that scatter-sum with attention inside each neighborhood: messages get queries and keys, and the center atom weighs its neighbors before aggregating. The center atom attends over its neighbors instead of over previous tokens, the same attention machinery constrained to respect rotation.

**Scale.** UMA is what happens when this recipe meets LLM-era training. The backbone is eSEN (the attention rung above), but the interesting delta is at our `self.readout` and everywhere a `Linear` lives: UMA swaps fixed weights for a Mixture of Linear Experts, the expert mixture chosen by a routing signal built from the dataset/task and the system's charge and spin. Our model has no idea whether a structure is a catalyst surface or a drug molecule; UMA is told, and picks its weights accordingly. The routing stays independent of positions, so it is decided once per structure rather than per step, and the chosen experts collapse to one linear map before the forward pass. Trained on roughly 500 million structures across five FAIR-Chemistry DFT datasets; UMA-medium is 1.4 billion parameters total but activates only about 50 million per structure. One checkpoint, whole periodic table, every domain in this catalog.

**Strictly local.** The last rung *removes* something. Our two message rounds give a 10 Å receptive field, and that is a distributed-computing liability: to update its boundary atoms, each GPU in a domain-decomposed simulation needs a ghost halo of neighbor data 10 Å deep from adjacent ranks, and every message round is a communication round. Allegro deletes message passing entirely (no `Interaction` layers, no information hops) and spends all its capacity on a deep equivariant function of each atom's *immediate* neighborhood. Receptive field equals the cutoff, full stop. Halos get thin and inter-GPU chatter drops, which is why strictly local models are attractive for the largest multi-node runs, billions of atoms across thousands of GPUs.

Six rungs, one contract. Elements and positions in, energy out, forces by autograd, and a backward pass in the serving loop all the way up the ladder.